# Baseline Evaluation

**Goal:** Evaluate the final DeBERTa-v3-large reward model on the held-out test set (1,000 examples) to establish our baseline performance.

This notebook visualizes the results generated by `src/evaluate.py`.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure results directory exists
os.makedirs('../results', exist_ok=True)

# Set publication-quality style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 150,
})

## 1. Load Metrics and Results

In [ ]:
# Load top-level metrics
with open('../results/baseline_metrics.json', 'r') as f:
    metrics = json.load(f)

print("Baseline Performance")
print(f"Test Loss:     {metrics['test_loss']:.4f}")
print(f"Test Accuracy: {metrics['test_accuracy'] * 100:.2f}%")

# Load instance-level results
df = pd.read_csv('../results/baseline_test_results.csv')
print(f"\nLoaded {len(df)} test instances.")

## 2. Reward Score Distribution

In [ ]:
plt.figure(figsize=(10, 6))

sns.kdeplot(data=df, x='chosen_score', fill=True, label='Chosen Reward', color='#7D3C98', alpha=0.7)
sns.kdeplot(data=df, x='rejected_score', fill=True, label='Rejected Reward', color='#CB4335', alpha=0.7)

plt.title('Distribution of Reward Scores on Test Set')
plt.xlabel('Scalar Reward')
plt.ylabel('Density')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig('../results/01(1)_figure_reward_distribution.png', bbox_inches='tight')
print("Saved figure to: results/01(1)_figure_reward_distribution.png")
plt.show()

## 3. Reward Gap (Margin)

The model correctly classifies a pair if the chosen score is strictly greater than the rejected score (i.e., gap > 0).

In [ ]:
df['reward_gap'] = df['chosen_score'] - df['rejected_score']

plt.figure(figsize=(10, 6))
sns.histplot(df['reward_gap'], bins=40, kde=True, color='#5D6D7E')

plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='Decision Boundary (0.0)')

plt.title('Reward Gap (Chosen - Rejected)')
plt.xlabel('Margin')
plt.ylabel('Count')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig('../results/01(2)_figure_reward_gap.png', bbox_inches='tight')
print("Saved figure to: results/01(2)_figure_reward_gap.png")
plt.show()

correct_pct = (df['reward_gap'] > 0).mean() * 100
print(f"Pairs with gap > 0: {correct_pct:.2f}% (Matches Accuracy)")

## 4. Key Finding

The model achieves a **59.80% test accuracy**. While better than random chance (50%), there is a significant **generalization gap** compared to the 75.68% training accuracy achieved during Kaggle compute.

This establishes our baseline. We will now investigate why the model fails (interpretability) and how easily these scores can be manipulated (adversarial robustness).